# Structured V_θ Landscape Sweep on TinyStories

## Goal

Systematically evaluate **every structured V_θ variant** (SQ1–SQ4 + parameter
variations) against the **MLP V_θ baseline** that achieved **26.42 PPL** on
TinyStories (P10g/P10h, `v_hidden=2048`, 16k steps, d=256, L=8).

All cells share the same PARFLM architecture — only V_θ is swapped.
This isolates the impact of the potential landscape parameterisation.

## Experiment cells

| Cell | V_θ variant | Key params | Why interesting |
|------|------------|------------|----------------|
| `A1` | **SQ3 K=4** (mixture, 4 wells) | ~167K | Matches PR2 basin count; Shakespeare parity |
| `A2` | **SQ3 K=8** (mixture, 8 wells) | ~333K | More basins; diminishing returns or improvement? |
| `A3` | **SQ3 K=16** (mixture, 16 wells) | ~663K | Over-parameterised structured; expressivity ceiling? |
| `A4` | **SQ1** (single diagonal well) | ~66K | Cheapest; can a single attractor suffice? |
| `A5` | **SQ2 r=4** (low-rank + diag) | ~198K | Off-diagonal correlations at low rank |
| `A6` | **SQ2 r=16** (low-rank + diag) | ~592K | Higher-rank off-diagonal |
| `A7` | **SQ4** (quad backbone + small MLP) | ~75K | Hybrid: structured base + learned correction |
| `A8` | **SQ4 large** (quad + wider MLP) | ~134K | More flexible hybrid |
| `A9` | **SQ3 K=4, tau=0.1** (cold mixture) | ~167K | Sharp basin selection (hard assignment limit) |
| `A10`| **SQ3 K=4, tau=10** (warm mixture) | ~167K | Soft blending (approaches single averaged well) |
| `B1` | **MLP baseline** (v_hidden=2048) | 9.4M | P10g/P10h reproduction reference |
| `B2` | **MLP small** (v_hidden=512) | ~660K | Can a smaller MLP match structured forms? |

## Baselines for comparison

- **MatchedGPT** (attention, d=256, 8L, 19.4M params): **7.81 PPL**
- **PARFLM MLP P10g** (v_hidden=2048, 22.6M params, 16k steps): **26.42 PPL**
- **Multi-ξ SPLM** (leak-free, 4k steps): **14.78 PPL**

## Run instructions

1. Set `CELL` to one of `A1`–`A10`, `B1`, or `B2`
2. Run all cells top to bottom
3. Results persist on GDrive in `semsimula_structured_vtheta_tinystories/`
4. After running all cells, run the comparison dashboard (last cell)


## 0. Environment setup + cell selector

In [ ]:
CELL = 'A1'       # one of: A1..A10, B1, B2
SEED = 0

REPO_URL        = 'https://github.com/dimitarpg13/semsimula-paper.git'
REPO_BRANCH     = 'main'
COLAB_REPO_PATH = '/content/semsimula-paper'
GDRIVE_OUT_REL  = 'semsimula_structured_vtheta_tinystories'

import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
print(f'IN_COLAB = {IN_COLAB}')


def _sh(cmd: str) -> None:
    print(f'$ {cmd}')
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        raise RuntimeError(f'command failed (exit {r.returncode}): {cmd}')


if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    GDRIVE_OUT = Path('/content/drive/MyDrive') / GDRIVE_OUT_REL
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)
    print(f'GDrive output root = {GDRIVE_OUT}')

    REPO_ROOT = Path(COLAB_REPO_PATH)
    if not (REPO_ROOT / '.git').exists():
        if REPO_ROOT.exists():
            shutil.rmtree(REPO_ROOT)
        _sh(
            f'git clone --depth 1 --branch {REPO_BRANCH} '
            f'{REPO_URL} {REPO_ROOT}'
        )
    else:
        try:
            _sh(f'git -C {REPO_ROOT} fetch --depth 1 origin {REPO_BRANCH}')
            _sh(f'git -C {REPO_ROOT} reset --hard origin/{REPO_BRANCH}')
        except RuntimeError as e:
            print(f'WARNING: could not refresh repo ({e}); using existing checkout.')

    DATA_CACHE = GDRIVE_OUT / 'data'
    DATA_CACHE.mkdir(exist_ok=True)
    repo_data_dir = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'data'
    if repo_data_dir.is_symlink():
        repo_data_dir.unlink()
    elif repo_data_dir.is_dir():
        shutil.rmtree(repo_data_dir)
    repo_data_dir.symlink_to(DATA_CACHE)
    print(f'data/ -> {DATA_CACHE}')

    _sh('pip install -q transformers huggingface_hub pyarrow')
else:
    REPO_ROOT = Path('.').resolve()
    while not (REPO_ROOT / '.git').exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent
    GDRIVE_OUT = REPO_ROOT / 'notebooks' / 'conservative_arch' / 'parf' / 'results' / 'structured_vtheta_tinystories'
    GDRIVE_OUT.mkdir(parents=True, exist_ok=True)

SARF_DIR = REPO_ROOT / 'notebooks' / 'conservative_arch'
sys.path.insert(0, str(SARF_DIR))
sys.path.insert(0, str(SARF_DIR / 'parf'))
sys.path.insert(0, str(SARF_DIR / 'sarf_mass_variant'))
sys.path.insert(0, str(SARF_DIR / 'energetic_minima'))
sys.path.insert(0, str(SARF_DIR / 'scaleup'))

RESULTS_ROOT = GDRIVE_OUT
RUN_DIR = RESULTS_ROOT / CELL / f'seed{SEED}'
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f'Run output dir = {RUN_DIR}')

## 1. GPU check

In [ ]:
import torch
import numpy as np

if torch.cuda.is_available():
    device = 'cuda'
    props = torch.cuda.get_device_properties(0)
    total_memory = props.total_memory / 1e9
    print(f'GPU: {props.name}  ({total_memory:.1f} GB)')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled (SPLM autograd.grad sensitivity)')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('Using MPS (Apple Silicon)')
else:
    device = 'cpu'
    print('WARNING: no GPU detected; training will be very slow')

print(f'device = {device}')
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

## 2. Experiment recipes

In [ ]:
RECIPES = {
    # ─── Structured V_theta variants ────────────────────────────────
    'A1': {
        'desc': 'SQ3 mixture K=4 (tau=1.0)',
        'v_theta_kind': 'sq3', 'K': 4, 'tau': 1.0,
        'rank': None,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'v_hidden': 2048, 'v_depth': 3,
        'lambda_v': 1e-2,
    },
    'A2': {
        'desc': 'SQ3 mixture K=8 (tau=1.0)',
        'v_theta_kind': 'sq3', 'K': 8, 'tau': 1.0,
        'rank': None,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'v_hidden': 2048, 'v_depth': 3,
        'lambda_v': 1e-2,
    },
    'A3': {
        'desc': 'SQ3 mixture K=16 (tau=1.0)',
        'v_theta_kind': 'sq3', 'K': 16, 'tau': 1.0,
        'rank': None,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'v_hidden': 2048, 'v_depth': 3,
        'lambda_v': 1e-2,
    },
    'A4': {
        'desc': 'SQ1 single diagonal well',
        'v_theta_kind': 'sq1', 'K': 1, 'tau': 1.0,
        'rank': None,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'v_hidden': 2048, 'v_depth': 3,
        'lambda_v': 1e-2,
    },
    'A5': {
        'desc': 'SQ2 low-rank+diag (rank=4)',
        'v_theta_kind': 'sq2', 'K': 1, 'tau': 1.0,
        'rank': 4,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'v_hidden': 2048, 'v_depth': 3,
        'lambda_v': 1e-2,
    },
    'A6': {
        'desc': 'SQ2 low-rank+diag (rank=16)',
        'v_theta_kind': 'sq2', 'K': 1, 'tau': 1.0,
        'rank': 16,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'v_hidden': 2048, 'v_depth': 3,
        'lambda_v': 1e-2,
    },
    'A7': {
        'desc': 'SQ4 hybrid quad + small MLP (h=32, depth=2)',
        'v_theta_kind': 'sq4', 'K': 1, 'tau': 1.0,
        'rank': None,
        'hybrid_v_hidden': 32, 'hybrid_v_depth': 2,
        'v_hidden': 2048, 'v_depth': 3,
        'lambda_v': 1e-2,
    },
    'A8': {
        'desc': 'SQ4 hybrid quad + wider MLP (h=128, depth=2)',
        'v_theta_kind': 'sq4', 'K': 1, 'tau': 1.0,
        'rank': None,
        'hybrid_v_hidden': 128, 'hybrid_v_depth': 2,
        'v_hidden': 2048, 'v_depth': 3,
        'lambda_v': 1e-2,
    },
    'A9': {
        'desc': 'SQ3 mixture K=4 COLD (tau=0.1)',
        'v_theta_kind': 'sq3', 'K': 4, 'tau': 0.1,
        'rank': None,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'v_hidden': 2048, 'v_depth': 3,
        'lambda_v': 1e-2,
    },
    'A10': {
        'desc': 'SQ3 mixture K=4 WARM (tau=10.0)',
        'v_theta_kind': 'sq3', 'K': 4, 'tau': 10.0,
        'rank': None,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'v_hidden': 2048, 'v_depth': 3,
        'lambda_v': 1e-2,
    },
    # ─── MLP baselines ─────────────────────────────────────────────
    'B1': {
        'desc': 'MLP baseline (v_hidden=2048, P10g reproduction)',
        'v_theta_kind': 'mlp', 'K': None, 'tau': None,
        'rank': None,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'v_hidden': 2048, 'v_depth': 3,
        'lambda_v': 0.0,
    },
    'B2': {
        'desc': 'MLP small (v_hidden=512)',
        'v_theta_kind': 'mlp', 'K': None, 'tau': None,
        'rank': None,
        'hybrid_v_hidden': None, 'hybrid_v_depth': None,
        'v_hidden': 512, 'v_depth': 3,
        'lambda_v': 0.0,
    },
}

if CELL not in RECIPES:
    raise ValueError(f'CELL must be one of {sorted(RECIPES)}; got {CELL!r}')

recipe = RECIPES[CELL]

# ─── Shared architecture (matches P10g/P10h exactly) ───────────
D              = 256
L              = 8
VOCAB_SIZE     = 50257
MAX_LEN        = 1024
DT             = 1.0
V_HIDDEN       = recipe['v_hidden']
V_DEPTH        = recipe['v_depth']
TOP_K          = 4
SCORE_HEAD_HIDDEN = 32
V_PHI_KIND     = 'structural_competitive'
V_PHI_PHI_HIDDEN = 64
V_PHI_THETA_HIDDEN = 64
V_PHI_MLP_HIDDEN = 32
V_PHI_D_TYPE   = 32
V_PHI_D_ANGLE  = 16
INIT_GAMMA     = 0.15
FIXED_GAMMA    = None
LAMBDA_V       = recipe['lambda_v']

STEPS          = 16000
BATCH          = 8       # micro-batch per accumulation step
GRAD_ACCUM     = 2       # effective batch = BATCH * GRAD_ACCUM = 16
BLOCK          = 512
LR             = 5e-4
WD             = 0.01
WARMUP         = 400
GRAD_CLIP      = 1.0
EVAL_INTERVAL  = 400
EVAL_ITERS     = 40
LOG_INTERVAL   = 50
GUMBEL_TAU_INIT = 1.0
GUMBEL_TAU_MIN  = 0.1

print(f'Cell {CELL}: {recipe["desc"]}')
print(f'  V_theta = {recipe["v_theta_kind"]}')
if recipe['K'] is not None:
    print(f'  K = {recipe["K"]}, tau = {recipe["tau"]}')
if recipe['rank'] is not None:
    print(f'  rank = {recipe["rank"]}')
if recipe['hybrid_v_hidden'] is not None:
    print(f'  hybrid MLP: hidden={recipe["hybrid_v_hidden"]}, depth={recipe["hybrid_v_depth"]}')
print(f'  lambda_V = {LAMBDA_V}')
print(f'  steps={STEPS}  batch={BATCH}  block={BLOCK}')

## 3. Load TinyStories

In [ ]:
from data_module import load_tiny_stories, get_batch

train_ids, val_ids = load_tiny_stories(max_train_tokens=5_000_000)
print(f'train: {len(train_ids):,} tokens   val: {len(val_ids):,} tokens')

## 4. Build model + V_θ hot-swap

In [ ]:
from model_parf_sparse import SparsePARFConfig, SparsePARFLM
from model_structured_vtheta import (
    MixtureQuadraticVTheta, QuadraticWellVTheta,
    LowRankQuadraticVTheta, HybridQuadraticVTheta,
    validate_analytical_grad,
)
from model_sarf_mass import causal_cumulative_mean
import torch.nn.functional as F_torch

SCALEUP_LOGFREQ = SARF_DIR / 'scaleup' / 'results' / 'logfreq_surprisal_tinystories.npy'
DRIVE_LOGFREQ = RESULTS_ROOT / 'logfreq_surprisal_tinystories.npy'

if SCALEUP_LOGFREQ.exists():
    LOGFREQ_PATH = SCALEUP_LOGFREQ
    print(f'Using bundled logfreq: {LOGFREQ_PATH}')
elif DRIVE_LOGFREQ.exists():
    LOGFREQ_PATH = DRIVE_LOGFREQ
    print(f'Using Drive-cached logfreq: {LOGFREQ_PATH}')
else:
    counts = np.bincount(train_ids.astype(np.int64), minlength=VOCAB_SIZE).astype(np.float64)
    p = (counts + 1.0) / (counts.sum() + VOCAB_SIZE)
    surprisal = (-np.log(p)).astype(np.float32)
    LOGFREQ_PATH = DRIVE_LOGFREQ
    LOGFREQ_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.save(LOGFREQ_PATH, surprisal)
    print(f'Built logfreq from train_ids; saved to {LOGFREQ_PATH}')

torch.manual_seed(SEED)

cfg = SparsePARFConfig(
    vocab_size=VOCAB_SIZE, d=D, max_len=MAX_LEN,
    L=L, v_hidden=V_HIDDEN, v_depth=V_DEPTH, dt=DT,
    v_phi_kind=V_PHI_KIND,
    v_phi_d_type=V_PHI_D_TYPE, v_phi_d_angle=V_PHI_D_ANGLE,
    v_phi_phi_hidden=V_PHI_PHI_HIDDEN,
    v_phi_theta_hidden=V_PHI_THETA_HIDDEN,
    v_phi_mlp_hidden=V_PHI_MLP_HIDDEN,
    mass_mode='logfreq',
    logfreq_path=str(LOGFREQ_PATH),
    init_gamma=INIT_GAMMA,
    causal_force=True,
    ln_after_step=True,
    ln_before_distance=True,
    per_layer_v_phi_scale=True,
    theta_activation='softsign',
    theta_form='bilinear',
    top_k=TOP_K,
    score_head_hidden=SCORE_HEAD_HIDDEN,
    gumbel_tau_init=GUMBEL_TAU_INIT,
    gumbel_tau_min=GUMBEL_TAU_MIN,
)
model = SparsePARFLM(cfg).to(device)

n_total_before = sum(p.numel() for p in model.parameters())
n_v_theta_before = sum(p.numel() for p in model.V_theta.parameters())
print(f'Before swap: total={n_total_before:,}  V_theta={n_v_theta_before:,}')

vkind = recipe['v_theta_kind']
if vkind == 'sq3':
    model.V_theta = MixtureQuadraticVTheta(
        d=D, K=recipe['K'], tau=recipe['tau'],
    ).to(device)
    print(f'[{CELL}] Swapped V_theta -> MixtureQuadraticVTheta(K={recipe["K"]}, tau={recipe["tau"]})')
    validate_analytical_grad(model.V_theta, d=D)
elif vkind == 'sq1':
    model.V_theta = QuadraticWellVTheta(d=D).to(device)
    print(f'[{CELL}] Swapped V_theta -> QuadraticWellVTheta')
    validate_analytical_grad(model.V_theta, d=D)
elif vkind == 'sq2':
    model.V_theta = LowRankQuadraticVTheta(
        d=D, rank=recipe['rank'],
    ).to(device)
    print(f'[{CELL}] Swapped V_theta -> LowRankQuadraticVTheta(rank={recipe["rank"]})')
    validate_analytical_grad(model.V_theta, d=D)
elif vkind == 'sq4':
    model.V_theta = HybridQuadraticVTheta(
        d=D,
        v_hidden=recipe['hybrid_v_hidden'],
        v_depth=recipe['hybrid_v_depth'],
    ).to(device)
    print(f'[{CELL}] Swapped V_theta -> HybridQuadraticVTheta('
          f'h={recipe["hybrid_v_hidden"]}, depth={recipe["hybrid_v_depth"]})')
    validate_analytical_grad(model.V_theta, d=D, tol=1e-4)
else:
    print(f'[{CELL}] Keeping MLP V_theta (v_hidden={V_HIDDEN}, v_depth={V_DEPTH})')

n_total = sum(p.numel() for p in model.parameters())
n_v_theta = sum(p.numel() for p in model.V_theta.parameters())
n_v_phi = sum(p.numel() for p in model.V_phi.parameters())
n_score = sum(p.numel() for p in model.score_head.parameters())

print(f'\nAfter swap:')
print(f'  total params    = {n_total:,}')
print(f'  V_theta params  = {n_v_theta:,}  ({n_v_theta/n_total*100:.1f}%)')
print(f'  V_phi params    = {n_v_phi:,}')
print(f'  score_head      = {n_score:,}')
print(f'  other (emb etc) = {n_total - n_v_theta - n_v_phi - n_score:,}')
print(f'\n  V_theta reduction vs MLP: {n_v_theta_before:,} -> {n_v_theta:,} '
      f'({n_v_theta_before/max(n_v_theta,1):.0f}x)')

HAS_ANALYTICAL_GRAD = callable(getattr(model.V_theta, 'analytical_grad', None))
IS_MIXTURE = vkind == 'sq3'
print(f'  analytical_grad available: {HAS_ANALYTICAL_GRAD}')

## 5. Causal probe (pre-training sanity check)

In [ ]:
from causal_probe_parf import assert_causal

try:
    assert_causal(model, vocab_size=VOCAB_SIZE, T=32, t_pert=20, seed=SEED)
    print(f'Causal probe PASSED')
except RuntimeError as exc:
    print(f'Causal probe FAILED: {exc}')
    raise

## 6. Training loop

In [ ]:
import math, time, json


def lr_at(step):
    if step < WARMUP:
        return LR * (step + 1) / WARMUP
    progress = (step - WARMUP) / max(STEPS - WARMUP, 1)
    return LR * 0.5 * (1.0 + math.cos(math.pi * min(progress, 1.0)))


def tau_at(step):
    anneal_fraction = 0.8
    warm = int((1.0 - anneal_fraction) * STEPS)
    if step < warm:
        return GUMBEL_TAU_INIT
    if step >= STEPS:
        return GUMBEL_TAU_MIN
    progress = (step - warm) / max(STEPS - warm, 1)
    return GUMBEL_TAU_INIT + (GUMBEL_TAU_MIN - GUMBEL_TAU_INIT) * min(progress, 1.0)


def forward_with_vreg(model, x, targets, lambda_v):
    """Standard PARF forward with optional V_theta regularisation."""
    h0 = model._embed(x)
    h_L, _ = model._stack_forward(h0, x, return_trajectory=False)

    logits = h_L @ model.E.weight.T
    loss_ntp = F_torch.cross_entropy(
        logits.reshape(-1, cfg.vocab_size),
        targets.reshape(-1),
    )

    v_reg_value = torch.tensor(0.0, device=x.device)
    if lambda_v > 0:
        xi = causal_cumulative_mean(h_L.detach())
        V_vals = model.V_theta(xi, h_L)
        v_reg_value = (V_vals ** 2).mean()
        loss = loss_ntp + lambda_v * v_reg_value
    else:
        loss = loss_ntp

    return logits, loss, loss_ntp, v_reg_value


@torch.no_grad()
def evaluate_model():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch(val_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        with torch.enable_grad():
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


opt = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR, betas=(0.9, 0.95), weight_decay=WD,
)
model.train()

log = []
best_ppl = float('inf')
t0 = time.time()

for step in range(STEPS):
    for g in opt.param_groups:
        g['lr'] = lr_at(step)
    model.set_gumbel_tau(tau_at(step))

    opt.zero_grad(set_to_none=True)
    step_loss_ntp = 0.0
    step_v_reg = 0.0
    step_loss_total = 0.0
    for _acc in range(GRAD_ACCUM):
        xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
        x = torch.from_numpy(xb).to(device)
        y = torch.from_numpy(yb).to(device)
        _, loss, loss_ntp, v_reg = forward_with_vreg(model, x, y, LAMBDA_V)
        (loss / GRAD_ACCUM).backward()
        step_loss_ntp   += loss_ntp.item() / GRAD_ACCUM
        step_v_reg      += v_reg.item()    / GRAD_ACCUM
        step_loss_total += loss.item()     / GRAD_ACCUM
    torch.nn.utils.clip_grad_norm_(
        [p for p in model.parameters() if p.requires_grad], GRAD_CLIP,
    )
    opt.step()

    if (step + 1) % LOG_INTERVAL == 0 or step == 0:
        elapsed = time.time() - t0
        tau_str = f'tau={tau_at(step):.3f}'
        print(f'[{CELL}] step {step + 1:>5}/{STEPS}  '
              f'lr={lr_at(step):.2e}  {tau_str}  '
              f'ntp={step_loss_ntp:.4f}  '
              f'v_reg={step_v_reg:.4f}  '
              f'total={step_loss_total:.4f}  '
              f'gamma={model.gamma.item():.3f}  '
              f'wall={elapsed:.0f}s')

    if (step + 1) % EVAL_INTERVAL == 0 or (step + 1) == STEPS:
        val_loss = evaluate_model()
        val_ppl = math.exp(val_loss)
        if val_ppl < best_ppl:
            best_ppl = val_ppl
        print(f'  >> val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}  '
              f'best_ppl={best_ppl:.2f}')
        log.append({
            'step': step + 1, 'val_loss': val_loss,
            'val_ppl': val_ppl, 'best_ppl': best_ppl,
            'train_loss_ntp': step_loss_ntp,
            'v_reg': step_v_reg, 'train_loss_total': step_loss_total,
            'lambda_v': LAMBDA_V,
            'gamma': model.gamma.item(),
        })

print(f'\n[{CELL}] Training done.  total wall = {time.time() - t0:.0f}s  '
      f'final val_ppl = {log[-1]["val_ppl"]:.2f}  '
      f'best_ppl = {best_ppl:.2f}')

## 7. Save checkpoint and logs

In [ ]:
from dataclasses import asdict

full_tag = f'structured_vtheta_{CELL}_{recipe["v_theta_kind"]}'
if recipe['K'] is not None:
    full_tag += f'_K{recipe["K"]}'
if recipe.get('tau') is not None and recipe['v_theta_kind'] == 'sq3':
    full_tag += f'_tau{recipe["tau"]:g}'
if recipe.get('rank') is not None:
    full_tag += f'_r{recipe["rank"]}'
if recipe.get('hybrid_v_hidden') is not None:
    full_tag += f'_h{recipe["hybrid_v_hidden"]}'
full_tag += f'_d{D}_L{L}_seed{SEED}'

ckpt_path = RUN_DIR / f'{full_tag}_ckpt_latest.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'model_cfg': asdict(cfg),
    'recipe': recipe,
    'cell': CELL,
    'best_ppl': best_ppl,
    'final_val_ppl': log[-1]['val_ppl'] if log else None,
    'step': STEPS,
    'seed': SEED,
    'n_params': n_total,
    'n_v_theta_params': n_v_theta,
    'n_v_phi_params': n_v_phi,
    'n_score_head_params': n_score,
    'has_analytical_grad': HAS_ANALYTICAL_GRAD,
    'elapsed_sec': time.time() - t0,
    'variant': f'structured_vtheta_{recipe["v_theta_kind"]}',
}, ckpt_path)
print(f'Checkpoint saved: {ckpt_path}')

log_path = RUN_DIR / f'{full_tag}_training_log.jsonl'
with open(log_path, 'w') as f:
    for entry in log:
        f.write(json.dumps(entry) + '\n')
print(f'Training log saved: {log_path}')

summary_path = RUN_DIR / f'{full_tag}_summary.md'
with open(summary_path, 'w') as f:
    f.write(f'# Training summary \u2014 {CELL}: {recipe["desc"]}\n\n')
    f.write(f'- V_theta: {recipe["v_theta_kind"]}')
    if recipe['K'] is not None:
        f.write(f' (K={recipe["K"]}')
        if recipe.get('tau') is not None:
            f.write(f', tau={recipe["tau"]}')
        f.write(')')
    if recipe.get('rank') is not None:
        f.write(f' (rank={recipe["rank"]})')
    f.write('\n')
    f.write(f'- lambda_V: {LAMBDA_V}\n')
    f.write(f'- params: total={n_total:,}  V_theta={n_v_theta:,}  '
            f'V_phi={n_v_phi:,}  score_head={n_score:,}\n')
    f.write(f'- d={D}  L={L}  max_len={MAX_LEN}  top_k={TOP_K}\n')
    f.write(f'- steps={STEPS}  batch={BATCH}  block={BLOCK}\n')
    f.write(f'- seed={SEED}\n')
    f.write(f'- elapsed: {time.time() - t0:.0f} s '
            f'({(time.time() - t0)/3600:.2f} h)\n')
    f.write(f'- analytical_grad: {HAS_ANALYTICAL_GRAD}\n')
    f.write(f'\n## Results\n\n')
    f.write(f'- Best val PPL: **{best_ppl:.2f}**\n')
    f.write(f'- Final val PPL: {log[-1]["val_ppl"]:.2f}\n')
    f.write(f'- Final gamma: {model.gamma.item():.4f}\n')
    f.write(f'\n## Context\n\n')
    f.write(f'| Reference | PPL |\n|---|---:|\n')
    f.write(f'| MatchedGPT (attention, 19.4M) | 7.81 |\n')
    f.write(f'| PARFLM MLP P10g (22.6M) | 26.42 |\n')
    f.write(f'| Multi-xi SPLM (leak-free) | 14.78 |\n')
    f.write(f'| **This run ({CELL}, {n_total:,} params)** | **{best_ppl:.2f}** |\n')
print(f'Summary saved: {summary_path}')

## 8. Training curve

In [ ]:
import matplotlib.pyplot as plt

steps_arr = [e['step'] for e in log]
ppls = [e['val_ppl'] for e in log]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(steps_arr, ppls, 'o-', label=f'{CELL}: {recipe["desc"]}',
        linewidth=2, markersize=4)

ax.axhline(7.81, color='red', linestyle='--', alpha=0.6,
           label='MatchedGPT (7.81)')
ax.axhline(26.42, color='orange', linestyle='--', alpha=0.6,
           label='PARFLM MLP P10g (26.42)')
ax.axhline(14.78, color='green', linestyle='--', alpha=0.6,
           label='Multi-xi SPLM (14.78)')

ax.set_xlabel('Training step')
ax.set_ylabel('Val PPL')
ax.set_title(f'{CELL}: {recipe["desc"]} (params: {n_total:,}, V_theta: {n_v_theta:,})')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RUN_DIR / f'{full_tag}_training.png', dpi=120)
plt.show()
print(f'best PPL = {best_ppl:.2f}')

## 9. V_θ landscape diagnostics

In [ ]:
v_samples = []
model.eval()
for _ in range(10):
    xb, _ = get_batch(val_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    with torch.enable_grad():
        h0 = model._embed(x)
        h_L, _ = model._stack_forward(h0, x, return_trajectory=False)
    h_L = h_L.detach()
    xi = causal_cumulative_mean(h_L)
    with torch.no_grad():
        V_vals = model.V_theta(xi, h_L).cpu().numpy().ravel()
    v_samples.append(V_vals)

V_all = np.concatenate(v_samples)
print(f'V_theta on real trajectories ({CELL}):')
print(f'  mean   = {V_all.mean():.4f}')
print(f'  std    = {V_all.std():.4f}')
print(f'  min    = {V_all.min():.4f}')
print(f'  max    = {V_all.max():.4f}')
print(f'  range  = {V_all.max() - V_all.min():.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(V_all, bins=100, color='#3a6ea5', alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', alpha=0.5)
ax.set_xlabel('V_\u03b8(\u03be, h)')
ax.set_ylabel('count')
ax.set_title(f'{CELL} V_\u03b8 distribution ({recipe["desc"]})')
plt.tight_layout()
plt.savefig(RUN_DIR / f'{full_tag}_v_theta_hist.png', dpi=120)
plt.show()

landscape_stats = {
    'mean': float(V_all.mean()), 'std': float(V_all.std()),
    'min': float(V_all.min()), 'max': float(V_all.max()),
    'range': float(V_all.max() - V_all.min()),
}
with open(RUN_DIR / f'{full_tag}_landscape_stats.json', 'w') as f:
    json.dump(landscape_stats, f, indent=2)
print(f'Landscape stats saved')

## 10. Attractor centre analysis (structured V_θ only)

In [ ]:
if IS_MIXTURE:
    print(f'=== Analytical attractor centres (SQ3, K={recipe["K"]}) ===')
    model.eval()
    try:
        from transformers import GPT2Tokenizer
        tok = GPT2Tokenizer.from_pretrained('gpt2')
        tok.pad_token = tok.eos_token
    except Exception:
        tok = None
        print('GPT2Tokenizer not available; skipping token-space projection')

    prompts = [
        'Once upon a time',
        'The cat sat on',
        'In a kingdom far away',
        'She opened the door and',
        'The little boy was very',
        'One day there was a big',
    ]
    attractor_data = {}
    for prompt in prompts:
        if tok is not None:
            ids = tok.encode(prompt, return_tensors='pt').to(device)
        else:
            xb, _ = get_batch(val_ids, 1, BLOCK, rng)
            ids = torch.from_numpy(xb).to(device)[:, :8]
        with torch.enable_grad():
            h0 = model._embed(ids)
            h_L, _ = model._stack_forward(h0, ids, return_trajectory=False)
        h_L = h_L.detach()
        xi = causal_cumulative_mean(h_L)
        with torch.no_grad():
            centres = model.V_theta.attractor_centres(xi)
            c_last = centres[0, -1, :, :]  # (K, d)

            if tok is not None:
                print(f'\n  Prompt: {prompt!r}')
                for k in range(recipe['K']):
                    scores = (c_last[k] @ model.E.weight.T).cpu()
                    top5 = scores.topk(5).indices.tolist()
                    top5_toks = [tok.decode([t]) for t in top5]
                    print(f'    Basin {k}: {top5_toks}')

            attractor_data[prompt] = {
                'centres': c_last.cpu().tolist(),
                'norms': [float(c_last[k].norm().item()) for k in range(recipe['K'])],
            }

    with open(RUN_DIR / f'{full_tag}_attractor_centres.json', 'w') as f:
        json.dump(attractor_data, f, indent=2)
    print(f'\nAttractor centres saved')

    fig, ax = plt.subplots(figsize=(8, 4))
    all_norms = []
    for p_data in attractor_data.values():
        all_norms.extend(p_data['norms'])
    ax.hist(all_norms, bins=30, color='#e07b39', alpha=0.7, edgecolor='white')
    ax.set_xlabel('||mu_k(xi)||')
    ax.set_ylabel('count')
    ax.set_title(f'{CELL}: Attractor centre norms (K={recipe["K"]})')
    plt.tight_layout()
    plt.savefig(RUN_DIR / f'{full_tag}_attractor_norms.png', dpi=120)
    plt.show()
else:
    print(f'Skipping attractor analysis (V_theta is {recipe["v_theta_kind"]}, not mixture)')

## 11. Step timing benchmark: analytical vs autograd force

In [ ]:
if device == 'cuda':
    print('=== Step timing benchmark ===')
    model.train()
    xb, yb = get_batch(train_ids, BATCH, BLOCK, rng)
    x = torch.from_numpy(xb).to(device)
    y = torch.from_numpy(yb).to(device)

    # Warmup
    for _ in range(3):
        _, loss = model(x, y)
        loss.backward()
        model.zero_grad(set_to_none=True)
    torch.cuda.synchronize()

    N_BENCH = 20
    torch.cuda.synchronize()
    t_start = time.time()
    for _ in range(N_BENCH):
        _, loss = model(x, y)
        loss.backward()
        model.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    t_end = time.time()

    ms_per_step = (t_end - t_start) / N_BENCH * 1000
    print(f'  {N_BENCH} steps: {t_end - t_start:.2f}s  '
          f'({ms_per_step:.0f} ms/step)')
    print(f'  analytical_grad: {HAS_ANALYTICAL_GRAD}')

    timing_data = {
        'cell': CELL, 'v_theta_kind': recipe['v_theta_kind'],
        'ms_per_step': ms_per_step, 'n_bench_steps': N_BENCH,
        'batch': BATCH, 'block': BLOCK,
        'has_analytical_grad': HAS_ANALYTICAL_GRAD,
        'gpu': torch.cuda.get_device_name(0),
    }
    with open(RUN_DIR / f'{full_tag}_timing.json', 'w') as f:
        json.dump(timing_data, f, indent=2)
    print(f'  Timing saved')
else:
    print('Skipping timing benchmark (no CUDA device)')

## 12. Cross-cell comparison dashboard

Run this cell after completing multiple experiment cells to see the full comparison table.

In [ ]:
ALL_CELLS = sorted(RECIPES.keys())

results = {}
for cell_name in ALL_CELLS:
    cell_dir = RESULTS_ROOT / cell_name / f'seed{SEED}'
    if not cell_dir.exists():
        results[cell_name] = None
        continue
    logs = sorted(cell_dir.glob('*_training_log.jsonl'))
    if not logs:
        results[cell_name] = None
        continue
    rows = [json.loads(line) for line in logs[-1].read_text().splitlines()]
    if not rows:
        results[cell_name] = None
        continue
    last = rows[-1]
    best = min(r['val_ppl'] for r in rows)
    ls_files = sorted(cell_dir.glob('*_landscape_stats.json'))
    ls = json.loads(ls_files[-1].read_text()) if ls_files else None
    timing_files = sorted(cell_dir.glob('*_timing.json'))
    timing = json.loads(timing_files[-1].read_text()) if timing_files else None
    ckpt_files = sorted(cell_dir.glob('*_ckpt_latest.pt'))
    n_params = None
    n_vt = None
    if ckpt_files:
        ckpt = torch.load(ckpt_files[-1], map_location='cpu', weights_only=False)
        n_params = ckpt.get('n_params')
        n_vt = ckpt.get('n_v_theta_params')
    results[cell_name] = {
        'desc': RECIPES[cell_name]['desc'],
        'best_ppl': best,
        'final_ppl': last['val_ppl'],
        'v_range': ls.get('range') if ls else None,
        'ms_step': timing.get('ms_per_step') if timing else None,
        'n_params': n_params,
        'n_vt': n_vt,
    }

header = (f'{"Cell":<6} {"Description":<42} {"V_theta":>10} '
          f'{"Total":>10} {"best PPL":>10} {"final PPL":>10} '
          f'{"V range":>8} {"ms/step":>8}')
print(header)
print('=' * len(header))

# Reference baselines
for label, ppl in [
    ('MatchedGPT (attention, 19.4M)', 7.81),
    ('PARFLM MLP P10g unreg (22.6M)', 26.42),
    ('Multi-xi SPLM leak-free', 14.78),
]:
    print(f'{"-":<6} {label:<42} {"-":>10} {"-":>10} '
          f'{ppl:>10.2f} {"-":>10} {"-":>8} {"-":>8}')
print('-' * len(header))

for cell_name in ALL_CELLS:
    r = results[cell_name]
    desc = RECIPES[cell_name]['desc']
    if r is None:
        print(f'{cell_name:<6} {desc:<42} {"\u2014":>10} {"\u2014":>10} '
              f'{"\u2014":>10} {"\u2014":>10} {"\u2014":>8} {"\u2014":>8}'
              f'  (not run)')
        continue
    vt = f"{r['n_vt']:,}" if r['n_vt'] else '\u2014'
    total = f"{r['n_params']:,}" if r['n_params'] else '\u2014'
    best = f"{r['best_ppl']:.2f}"
    final = f"{r['final_ppl']:.2f}"
    vr = f"{r['v_range']:.1f}" if r['v_range'] is not None else '\u2014'
    ms = f"{r['ms_step']:.0f}" if r['ms_step'] is not None else '\u2014'
    print(f'{cell_name:<6} {desc:<42} {vt:>10} {total:>10} '
          f'{best:>10} {final:>10} {vr:>8} {ms:>8}')

print()
completed = [c for c in ALL_CELLS if results[c] is not None]
print(f'Completed: {len(completed)}/{len(ALL_CELLS)} cells')
if completed:
    best_cell = min(completed, key=lambda c: results[c]['best_ppl'])
    print(f'Best cell so far: {best_cell} -> '
          f'{results[best_cell]["best_ppl"]:.2f} PPL '
          f'({results[best_cell]["desc"]})')